# MBPP EDA

- MBPP (Mostly Basic Python Programming)
- 출처 : https://huggingface.co/datasets/google-research-datasets/mbpp   
- 설명 : 구글 리서치(Google Research)에서 2021년에 발표한 Program Synthesis분야 데이터로, 약 1,000개의 크라우드소싱된 Python 프로그래밍 문제로 구성되어 있으며, 각 문제는 문제 설명, 코드 솔루션, 테스트 케이스를 포함하고 있다.
- 평가지표 : pass@1

*mbpp - sanitized(test) 257 rows

## overview :
- ch01 : 데이터 톺아보기
- ch02 : task 활용 관점으로 데이터 분석

---
## ch01 : 데이터 톺아보기   

01. 데이터 확인
02. 한 샘플 자세히 보기
03. **동작 흐름 관점 분석**
04. 연구 관점 정리

### 1. 데이터 확인하기

In [1]:
# 1. 데이터 확인
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("../../datasets/mbpp_sanitized/raw/mbpp_sanitized.jsonl")

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()

,source_file,task_id,prompt,code,test_imports,test_list
0,Benchmark Questions Verification V2.ipynb,11,Write a python function to remove first and la...,"def remove_Occ(s,ch): \n for i in range(len...",[],"[assert remove_Occ(""hello"",""l"") == ""heo"", asse..."
1,Benchmark Questions Verification V2.ipynb,12,Write a function to sort a given matrix in asc...,"def sort_matrix(M):\n result = sorted(M, ke...",[],"[assert sort_matrix([[1, 2, 3], [2, 4, 5], [1,..."
2,Benchmark Questions Verification V2.ipynb,14,Write a python function to find the volume of ...,"def find_Volume(l,b,h) : \n return ((l * b ...",[],"[assert find_Volume(10,8,6) == 240, assert fin..."
3,Benchmark Questions Verification V2.ipynb,16,Write a function to that returns true if the i...,import re\ndef text_lowercase_underscore(text)...,[],"[assert text_lowercase_underscore(""aab_cbbbc"")..."
4,Benchmark Questions Verification V2.ipynb,17,Write a function that returns the perimeter of...,def square_perimeter(a):\n perimeter=4*a\n r...,[],"[assert square_perimeter(10)==40, assert squar..."


In [2]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.dtypes

shape: (257, 6)
columns: ['source_file', 'task_id', 'prompt', 'code', 'test_imports', 'test_list']


source_file     object
task_id          int64
prompt          object
code            object
test_imports    object
test_list       object
dtype: object

### 1-2. 데이터 필드 확인하기   
출처 : https://huggingface.co/datasets/google-research-datasets/mbpp  

> Data Fields:   
> - source_file: 데이터 출처
> - text/prompt: 모델에 입력되는 문제 설명
> - code: 정답 코드(참고용)
> - test_setup_code/test_imports: 테스트 실행 전에 피료한 코드(라이브러리, 환경 설정 등)
> - test_list: 정답 여부를 판단하는 테스트 코드 리스트(검증 조건)
> - challenge_test_list: 더 어려운 검증용 테스트(robustness)

### 2. 샘플 데이터 자세히 뜯어보기

In [3]:
# 샘플 1개에 대해 확인하기
sample = df.iloc[0].to_dict()
for k, v in sample.items():
    print(f"\n===== {k} =====")
    print(v if isinstance(v, str) else repr(v))


===== source_file =====
Benchmark Questions Verification V2.ipynb

===== task_id =====
11

===== prompt =====
Write a python function to remove first and last occurrence of a given character from the string.

===== code =====
def remove_Occ(s,ch): 
    for i in range(len(s)): 
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    for i in range(len(s) - 1,-1,-1):  
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    return s 

===== test_imports =====
[]

===== test_list =====
['assert remove_Occ("hello","l") == "heo"', 'assert remove_Occ("abcda","a") == "bcd"', 'assert remove_Occ("PHP","P") == "H"']


### 3. 동작 흐름 관점 분석

한 샘플의 동작 :    
(1) input : text  

(2) output code : 모델이 함수 전체 생성  
    → code = 함수 이름, 파라미터, 클래스, 로직을 다 생성해야함  

(3) exec(code)   ---------------------- **Structural failure**   
    → 함수 생성

(4) exec(test_setup_code)  
    → check 함수 생성(import 등 환경 설정)  

(5) test_list 실행   ---------------------- **Semantic failure**   
    → for t in test_list: exec(t)


(1) input : text   
**모델 입력값 :  자연어 입력** 

In [4]:
# ===== text =====
# Write a function to find the longest chain which can be formed from the given set of pairs.

### 4. 연구 포인트

MBPP는 '코드를 생성하는 문제'가 아니라 자연어로 입력했을 때 '실행 가능한 코드를 생성해서 테스트를 통과하는 문제'로 보임

---
## ch02 : 데이터 톺아보기   

01. 전체 길이 분석
02. input/output 관점
03. Test 구조 분석
04. Research implication 

### 1. 길이 관점

In [5]:
# text_len : 모델에 들어가는 길이
# code_len : 모델이 출력하는 길이 관점 추정
# test_list_len : 평가 복잡도? 검증 조건의 길이

# 길이 (문자 기준)
for col in ["prompt", "code", "test_list", "test_imports", ]:
    df[f"{col}_len"] = df[col].astype(str).str.len()

# 개수 (리스트 기준)
df["test_list_count"] = df["test_list"].apply(len)
df["test_imports_count"] = df["test_imports"].apply(len)

# 확인
df[[
    "prompt_len",
    "code_len",
    "test_list_len",
    "test_list_count",
    'test_imports_len',
    'test_imports_count',
]].describe()

,prompt_len,code_len,test_list_len,test_list_count,test_imports_len,test_imports_count
count,257.000000,257.000000,257.000000,257.000000,257.000000,257.000000
mean,86.988327,157.105058,198.474708,3.019455,2.657588,0.050584
std,33.554466,115.469831,185.199113,0.164206,2.854456,0.219574
min,39.000000,36.000000,74.000000,3.000000,2.000000,0.000000
25%,65.000000,90.000000,117.000000,3.000000,2.000000,0.000000
50%,81.000000,124.000000,161.000000,3.000000,2.000000,0.000000
75%,97.000000,185.000000,234.000000,3.000000,2.000000,0.000000
max,252.000000,906.000000,2631.000000,5.000000,15.000000,1.000000


In [6]:
preview_cols = ['task_id', 'prompt', 'code', 'test_list', 'test_imports']
df[preview_cols].head(5)

,task_id,prompt,code,test_list,test_imports
0,11,Write a python function to remove first and la...,"def remove_Occ(s,ch): \n for i in range(len...","[assert remove_Occ(""hello"",""l"") == ""heo"", asse...",[]
1,12,Write a function to sort a given matrix in asc...,"def sort_matrix(M):\n result = sorted(M, ke...","[assert sort_matrix([[1, 2, 3], [2, 4, 5], [1,...",[]
2,14,Write a python function to find the volume of ...,"def find_Volume(l,b,h) : \n return ((l * b ...","[assert find_Volume(10,8,6) == 240, assert fin...",[]
3,16,Write a function to that returns true if the i...,import re\ndef text_lowercase_underscore(text)...,"[assert text_lowercase_underscore(""aab_cbbbc"")...",[]
4,17,Write a function that returns the perimeter of...,def square_perimeter(a):\n perimeter=4*a\n r...,"[assert square_perimeter(10)==40, assert squar...",[]


### 2. intput/output 구조 관찰

In [7]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'prompt'])}\n")
    print(df.loc[i, "prompt"])


================ SAMPLE 0 ================ len : 97

Write a python function to remove first and last occurrence of a given character from the string.

================ SAMPLE 1 ================ len : 92

Write a function to sort a given matrix in ascending order according to the sum of its rows.

================ SAMPLE 2 ================ len : 65

Write a python function to find the volume of a triangular prism.


In [8]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'code'])}\n")
    print(df.loc[i, "code"])


================ SAMPLE 0 ================ len : 269

def remove_Occ(s,ch): 
    for i in range(len(s)): 
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    for i in range(len(s) - 1,-1,-1):  
        if (s[i] == ch): 
            s = s[0 : i] + s[i + 1:] 
            break
    return s 

================ SAMPLE 1 ================ len : 69

def sort_matrix(M):
    result = sorted(M, key=sum)
    return result

================ SAMPLE 2 ================ len : 55

def find_Volume(l,b,h) : 
    return ((l * b * h) / 2) 


### 3. test 구조 관찰

In [9]:
for i in range(3):
    print(f"\n================ TEST {i} - {df.loc[i, 'task_id']} ================ len : {len(df.loc[i, 'test_list'])}\n")
    print(df.loc[i, "test_list"])


================ TEST 0 - 11 ================ len : 3

['assert remove_Occ("hello","l") == "heo"', 'assert remove_Occ("abcda","a") == "bcd"', 'assert remove_Occ("PHP","P") == "H"']

================ TEST 1 - 12 ================ len : 3

['assert sort_matrix([[1, 2, 3], [2, 4, 5], [1, 1, 1]])==[[1, 1, 1], [1, 2, 3], [2, 4, 5]]', 'assert sort_matrix([[1, 2, 3], [-2, 4, -5], [1, -1, 1]])==[[-2, 4, -5], [1, -1, 1], [1, 2, 3]]', 'assert sort_matrix([[5,8,9],[6,4,3],[2,1,4]])==[[2, 1, 4], [6, 4, 3], [5, 8, 9]]']

================ TEST 2 - 14 ================ len : 3

['assert find_Volume(10,8,6) == 240', 'assert find_Volume(3,2,2) == 6', 'assert find_Volume(1,2,1) == 1']


### 4. 연구관점 정리

MBPP는 모델이 함수 구조와 로직을 동시에 생성해야 하기 때문에,HumanEval보다 structural generation burden이 더 크다.